In [1]:
import sys, tensorflow as tf
print("Python exe:", sys.executable)
print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))




# In a notebook cell that’s running on the tf_310 kernel:

%pip uninstall -y scikit-image

# Install a NumPy-2 compatible scikit-image (0.25+). Pull a fresh wheel, no cache.
%pip install --no-cache-dir --upgrade "scikit-image==0.25.2"

# (Optional fallback if you STILL see the dtype error — build from source)
# %pip install --no-cache-dir --force-reinstall --no-binary=:all: "scikit-image==0.25.2"

# Verify everything lines up
import numpy as np, skimage, skimage.measure as measure
print("NumPy:", np.__version__, "scikit-image:", skimage.__version__)
print("label smoke test:", measure.label(np.zeros((4,4), dtype=np.uint8)).shape)



2025-09-16 10:29:23.138779: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Python exe: /home/rbielski/miniconda3/envs/tf_310/bin/python
TF version: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Found existing installation: scikit-image 0.25.2
Uninstalling scikit-image-0.25.2:
  Successfully uninstalled scikit-image-0.25.2
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 35.8 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.
NumPy: 2.2.1 scikit-image: 0.25.2
label smoke test: (4, 4)


No GPUs found; training will run on CPU.


In [ ]:
import random
import shutil
from pathlib import Path



# ------------------------------------------------------------------
# Paths and parameters
# ------------------------------------------------------------------
images_dir = Path('/home/rbielski/Atlas_2/Training/Images')
masks_dir  = Path('/home/rbielski/Atlas_2/Training/Masks')
output_root = Path('/home/rbielski/Atlas_2/Training_Split')  # sibling to Training/

train_frac = 0.8  # 80% of pairs for training
seed = 42

# Tokens indicating modality or mask descriptors
image_tokens = ['_T1w', '_T1', '_t1', '_T2w', '_t2',
                '_flair', '_FLAIR', '_dwi', '_DWI',
                '_adc', '_ADC', '_image', '_brain']
mask_tokens  = ['_mask', '_lesion', '_label', '_seg', '_desc']

def strip_at_first_token(name, tokens):
    indices = [name.find(tok) for tok in tokens if tok in name]
    return name[:min(indices)] if indices else name

def find_pairs(images_dir, masks_dir):
    """Identify image–mask pairs by matching shared prefixes."""
    image_files = list(images_dir.rglob('*.nii.gz'))
    mask_files  = list(masks_dir.rglob('*.nii.gz'))
    pairs = []
    for mask_path in mask_files:
        mask_base = strip_at_first_token(mask_path.stem, mask_tokens)
        match = None
        for img_path in image_files:
            img_base = strip_at_first_token(img_path.stem, image_tokens)
            if img_base == mask_base or img_base in mask_base or mask_base in img_base:
                match = img_path
                break
        if match:
            pairs.append((match, mask_path))
    return pairs

def split_pairs(pairs, train_frac=0.8, seed=42):
    random.seed(seed)
    pairs_shuffled = pairs.copy()
    random.shuffle(pairs_shuffled)
    n_train = int(len(pairs_shuffled) * train_frac)
    return pairs_shuffled[:n_train], pairs_shuffled[n_train:]

def copy_pairs(pairs, dest_images: Path, dest_masks: Path):
    dest_images.mkdir(parents=True, exist_ok=True)
    dest_masks.mkdir(parents=True, exist_ok=True)
    for img_path, mask_path in pairs:
        shutil.copy2(img_path, dest_images / img_path.name)
        shutil.copy2(mask_path, dest_masks / mask_path.name)

# ------------------------------------------------------------------
# Execute the splitting
# ------------------------------------------------------------------
if not images_dir.exists() or not masks_dir.exists():
    raise FileNotFoundError("Could not find the specified Images or Masks directories.")

pairs = find_pairs(images_dir, masks_dir)
if not pairs:
    raise RuntimeError("No image–mask pairs could be identified. Check your filenames.")

train_pairs, test_pairs = split_pairs(pairs, train_frac=train_frac, seed=seed)

# Create split structure under output_root
train_img_dir = output_root / 'Training_Set' / 'Images'
train_msk_dir = output_root / 'Training_Set' / 'Masks'
test_img_dir  = output_root / 'Test_set'    / 'Images'
test_msk_dir  = output_root / 'Test_set'    / 'Masks'

print(f'Copying {len(train_pairs)} pairs into {train_img_dir.parent}…')
copy_pairs(train_pairs, train_img_dir, train_msk_dir)

print(f'Copying {len(test_pairs)} pairs into {test_img_dir.parent}…')
copy_pairs(test_pairs, test_img_dir, test_msk_dir)

print('✅ Structured dataset split complete.')


SyntaxError: unterminated string literal (detected at line 19) (3926566403.py, line 19)

In [4]:
# ==== 3D MRI + mask quick viewer (Training_Set only) =========================
# Paths
from pathlib import Path
IMAGES_DIR = Path("/home/rbielski/Atlas_2/Training_Split/Training_Set/Images")
MASKS_DIR  = Path("/home/rbielski/Atlas_2/Training_Split/Training_Set/Masks")

import os, math, logging
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from functools import lru_cache
import ipywidgets as W
from IPython.display import display, clear_output

# Quiet down nibabel "qfac" chatter
logging.getLogger("nibabel").setLevel(logging.ERROR)

# ---------------- pairing helpers (fit your filenames) -----------------------
def _img_id(name: str) -> str:
    """ID from image file name, e.g.
    sub-xxx_ses-1_space-..._T1w.nii.gz  ->  sub-xxx_ses-1_space-...
    """
    base = name
    if base.endswith(".nii.gz"): base = base[:-7]
    # strip common image suffixes at the end
    for suf in ["_T1w", "_t1", "_T2w", "_FLAIR", "_image", "_brain"]:
        if base.endswith(suf):
            base = base[: -len(suf)]
            break
    return base

def _mask_id(name: str) -> str:
    """ID from mask file name, e.g.
    sub-xxx_ses-1_space-..._label-L_desc-T1lesion_mask.nii.gz
      -> sub-xxx_ses-1_space-...
    """
    base = name
    if base.endswith(".nii.gz"): base = base[:-7]
    for tok in ["_label", "_lesion", "_mask", "_seg"]:
        i = base.find(tok)
        if i != -1:
            base = base[:i]
            break
    return base

def build_pairs(images_dir: Path, masks_dir: Path):
    imgs = { _img_id(p.name): p for p in sorted(images_dir.glob("*.nii.gz")) }
    msks = { _mask_id(p.name): p for p in sorted(masks_dir.glob("*.nii.gz")) }
    common = sorted(set(imgs).intersection(msks))
    pairs = [(imgs[k], msks[k]) for k in common]
    return pairs, len(imgs), len(msks), len(common)

pairs, n_img, n_msk, n_pair = build_pairs(IMAGES_DIR, MASKS_DIR)

print(f"Found images: {n_img} | masks: {n_msk} | paired: {n_pair}")
if n_pair == 0:
    raise RuntimeError(
        "No pairs found in Training_Set. Check that files exist in:\n"
        f"- {IMAGES_DIR}\n- {MASKS_DIR}\n"
        "and that image IDs (before _T1w) match mask IDs (before _label/_mask)."
    )

# ---------------- caching loaders & utilities --------------------------------
@lru_cache(maxsize=64)
def _load_nii(path: str):
    img = nib.load(path)
    data = img.get_fdata()
    return data  # float64/float32 depending on file

def _norm01(x, invert=False):
    x = np.asarray(x, dtype=np.float32)
    # robust [p2, p98] window
    p2, p98 = np.percentile(x[np.isfinite(x)], [2, 98])
    if p98 <= p2:
        p2, p98 = x.min(), x.max()
    x = np.clip((x - p2) / max(1e-6, (p98 - p2)), 0, 1)
    if invert: x = 1.0 - x
    return x

def _slice2d(vol, axis, idx):
    if axis == 2:   # axial (z)
        return vol[:, :, idx]
    elif axis == 1: # coronal (y)
        return vol[:, idx, :]
    else:           # sagittal (x)
        return vol[idx, :, :]

def _edges2d(m):
    # simple 2D edge mask via dilation difference (fast)
    from scipy.ndimage import binary_dilation
    m = m.astype(bool)
    return binary_dilation(m) & (~m)

# ---------------- widgets -----------------------------------------------------
split_label = W.HTML(f"<b>Training_Set only</b> — {n_pair} pairs found")

axis_rb = W.RadioButtons(
    options=[("Axial (z)", 2), ("Coronal (y)", 1), ("Sagittal (x)", 0)],
    value=2, description="Axis:"
)

# Dropdown options: nice label, real (img,mask) tuple as value
def _option_label(img_path, msk_path):
    # show the shared ID (before suffix)
    return os.path.basename(msk_path).split("_label")[0]

pair_dd = W.Dropdown(
    options=[(_option_label(i, m), (str(i), str(m))) for (i, m) in pairs],
    description="Pair:",
    layout=W.Layout(width="100%")
)

slice_sl = W.IntSlider(description="Slice:", min=0, max=1, value=0, continuous_update=False)

mask_alpha = W.FloatSlider(description="Mask α:", min=0.0, max=1.0, step=0.05, value=0.55)
edges_only = W.Checkbox(description="Edges only (faster/clearer)", value=True)
invert_img = W.Checkbox(description="Invert image", value=False)
status = W.HTML("Viewer ready.")

controls = W.VBox([
    split_label,
    W.HBox([axis_rb, slice_sl]),
    pair_dd,
    W.HBox([mask_alpha, edges_only, invert_img]),
    status,
])

out = W.Output()

# ---------------- reactive update --------------------------------------------
def _update_slider_range(*_):
    img_path, msk_path = pair_dd.value
    vol = _load_nii(img_path)
    ax  = axis_rb.value
    max_idx = int(vol.shape[ax] - 1)
    slice_sl.max = max(0, max_idx)
    # keep current value in range
    slice_sl.value = min(slice_sl.value, slice_sl.max)

def _redraw(*_):
    with out:
        clear_output(wait=True)
        try:
            img_path, msk_path = pair_dd.value
            vol = _load_nii(img_path)
            msk = _load_nii(msk_path)
            ax  = axis_rb.value
            idx = int(slice_sl.value)

            if vol.shape[:3] != msk.shape[:3]:
                status.value = (f"<span style='color:#e55'>Shape mismatch: "
                                f"{vol.shape[:3]} vs {msk.shape[:3]}</span>")
            else:
                status.value = " "

            img2d = _slice2d(vol, ax, idx)
            m2d   = _slice2d(msk, ax, idx) > 0

            img2d = _norm01(img2d, invert=invert_img.value)

            plt.figure(figsize=(6,6))
            plt.imshow(img2d.T, cmap="gray", origin="lower")
            if edges_only.value:
                e = _edges2d(m2d)
                plt.contour(e.T, levels=[0.5], linewidths=0.7, colors="r")
            else:
                plt.imshow(np.ma.masked_where(~m2d.T, m2d.T), cmap="jet", alpha=float(mask_alpha.value), origin="lower")
            plt.axis("off")
            plt.show()
        except Exception as e:
            status.value = f"<span style='color:#e55'>Error: {e}</span>"

# wire up events
pair_dd.observe(_update_slider_range, names="value")
pair_dd.observe(_redraw, names="value")
axis_rb.observe(_update_slider_range, names="value")
axis_rb.observe(_redraw, names="value")
slice_sl.observe(_redraw, names="value")
invert_img.observe(_redraw, names="value")
edges_only.observe(_redraw, names="value")
mask_alpha.observe(_redraw, names="value")

# initial slider range & draw
_update_slider_range()
_redraw()

display(controls, out)
# ============================================================================== 


Found images: 524 | masks: 524 | paired: 524


Output()

In [7]:
from pathlib import Path
import nibabel as nib
import numpy as np

def estimate_mask_loss_due_to_target(pairs, target_shape, top_k=10):
    """
    For each (img, mask) pair, simulate the center crop/pad to target_shape and
    compute how many positive mask voxels would be lost. Prints summary.
    """
    losses = []
    examples = []
    for img_p, mask_p in pairs:
        try:
            mask = nib.load(str(mask_p)).get_fdata().astype(np.float32)
            mask_bin = (mask > 0.5).astype(np.uint8)
            orig_pos = int(mask_bin.sum())

            # Compute center crop slices from the MASK SHAPE (same as IMG in your set)
            in_slices = compute_center_slices(mask_bin.shape, target_shape)
            kept = mask_bin[in_slices[0], in_slices[1], in_slices[2]]
            kept_pos = int(kept.sum())
            lost = orig_pos - kept_pos
            frac = (lost / orig_pos) if orig_pos > 0 else 0.0

            losses.append(frac)
            if lost > 0:
                examples.append((float(frac), str(mask_p)))
        except Exception as e:
            print(f"[warn] failed on {mask_p}: {e}")

    if not losses:
        print("No masks found to audit.")
        return

    losses = np.array(losses, dtype=np.float32)
    any_loss = (losses > 0).sum()
    print(f"\n📏 Target shape = {target_shape}")
    print(f"Pairs audited: {len(losses)}")
    print(f"Pairs with ANY lesion voxels lost by center-crop: {any_loss} ({any_loss/len(losses)*100:.1f}%)")
    if any_loss > 0:
        print(f"Loss fraction stats (only over masks with >0 voxels): "
              f"mean={losses.mean():.4f}, median={np.median(losses):.4f}, max={losses.max():.4f}")
        examples.sort(reverse=True)
        print("\nTop examples (fraction lost, mask path):")
        for frac, path in examples[:top_k]:
            print(f"  {frac:.4f}  {path}")
    else:
        print("✅ No lesion voxels are lost by the center crop to target.")

# ---- run it (use your already-built `train_pairs` from the loader/split) ----
# If you don't have `train_pairs` in scope yet, quickly rebuild pairs:
pairs, _ = load_generic_dataset(DynamicTrainingConfig())
target_shape = DynamicTrainingConfig().INPUT_SHAPE[:-1]  # after detect_input_shape call in your run, substitute that tuple here if needed

# If you've already computed and logged INPUT_SHAPE earlier, plug it in explicitly, e.g.:
# target_shape = (192, 240, 192)

estimate_mask_loss_due_to_target(pairs, target_shape, top_k=8)


2025-09-16 13:06:35,501 - SmartSOTA_Dynamic - INFO - 📋 DynamicTrainingConfig initialised:
   Data directory: /home/rbielski/Atlas_2/Training_Split/Training_Set
   Batch size: 2
   Warmup epochs: 15
   Minimum LR: 5e-07
   Dice/boundary weights: 0.4:0.6

2025-09-16 13:06:35,502 - SmartSOTA_Dynamic - INFO - 📚 Loading generic dataset (RB pairing rules)...
2025-09-16 13:06:35,503 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=7.41GB | GPU mem tracking failed | Disk: 1741.3GB free
2025-09-16 13:06:35,510 - SmartSOTA_Dynamic - INFO - ✅ Found 524 images under /home/rbielski/Atlas_2/Training_Split/Training_Set/Images
2025-09-16 13:06:35,511 - SmartSOTA_Dynamic - INFO - ✅ Found 524 masks under  /home/rbielski/Atlas_2/Training_Split/Training_Set/Masks
2025-09-16 13:09:52,281 - SmartSOTA_Dynamic - INFO - 📊 Created 524 image–mask pairs
2025-09-16 13:09:52,282 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-09-16 13:09:52,283 - SmartSOTA_Dynamic - INFO - Memory at datas

TypeError: 'NoneType' object is not subscriptable

In [ ]:

"""
SMART SOTA 2025: Stroke Lesion Segmentation (Dynamic Input Version)

This training script builds upon prior production and cropped variants but adds
support for arbitrary volumetric input sizes.  It automatically determines
the largest spatial dimensions present in a dataset and pads smaller volumes
so that all inputs share the same shape.  The script retains detailed
logging, memory monitoring, data augmentation and custom layers from the
previous versions while incorporating recommendations from the latest model
evaluation:

* Dice/boundary loss weights adjusted to emphasise boundary precision
* Over‑segmentation mitigation via adjustable decision threshold
* Slightly stronger augmentation (rotations/flips/gamma) when overfitting
  is suspected
* Tunable L2 regularisation and dropout rates
* Longer warm‑up and lower minimum learning rate

The batch size is fixed at 2, but input volumes may be of any shape as long
as the corresponding mask has identical dimensions.  All three spatial
dimensions are padded to the maximum observed size for consistent training.
"""

import os
import sys
import logging
from pathlib import Path

# ---- Environment (set BEFORE importing TensorFlow) ----
import os

# Keep: quiet logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Optional: better GPU allocator (helps reduce fragmentation on long runs)
# Works with TF 2.10+ built for CUDA 11/12.
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Don't set for normal training:
# - CUDA_LAUNCH_BLOCKING=1  # debug-only; forces sync and can make training very slow
# - TF_XLA_FLAGS / XLA_FLAGS  # generally unnecessary on TF 2.20; can cause confusion
# - TF_ENABLE_ONEDNN_OPTS=0  # controls CPU-only kernels; leave default unless you need bit-for-bit CPU numerics


import tensorflow as tf

# See GPUs and enable memory growth (good practice)
gpus = tf.config.list_physical_devices("GPU")
print("Visible GPUs:", gpus)
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print(f"Could not set memory growth on {gpu}: {e}")

# Optional: use all visible GPUs
strategy = tf.distribute.MirroredStrategy() if gpus else tf.distribute.get_strategy()
print("Strategy:", type(strategy).__name__)

# Build/compile inside the scope if you use strategy
# with strategy.scope():
#     model = ...
#     model.compile(...)



# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------
# Use two file handlers and one stream handler.  One file captures all
# high‑level events (INFO and above) and the other captures per‑process
# debugging output.  A console stream is kept for quick feedback.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('logs/smart_sota_dynamic.log'),
        logging.FileHandler(f'logs/training_dynamic_{os.getpid()}.debug.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('SmartSOTA_Dynamic')
logger.setLevel(logging.DEBUG)

# ---------------------------------------------------------------------------
# Imports with graceful degradation
# ---------------------------------------------------------------------------
try:
    import tensorflow as tf
    from tensorflow.keras import layers
    import numpy as np
    import nibabel as nib
    from scipy.ndimage import zoom, binary_dilation, rotate
    from sklearn.model_selection import KFold
    from skimage import measure
    import json
    import time
    import math
    import random
    import gc
    import psutil
    from functools import lru_cache
    from concurrent.futures import ThreadPoolExecutor
    logger.info("✅ All imports successful")
except ImportError as e:
    logger.critical(f"❌ Import failed: {e}")
    sys.exit(1)

# ---------------------------------------------------------------------------
# Runtime configuration
# ---------------------------------------------------------------------------
# Disable eager execution for performance and to avoid certain CUDNN issues.
tf.config.run_functions_eagerly(False)
logger.info(f"TensorFlow eager execution: {tf.executing_eagerly()}")

warnings_to_ignore = [UserWarning, DeprecationWarning, FutureWarning]
for w in warnings_to_ignore:
    tf.autograph.set_verbosity(0)
import warnings
for w in warnings_to_ignore:
    warnings.filterwarnings("ignore", category=w)
warnings.filterwarnings("ignore", module="nibabel")

logger.info(
    f"Environment verified:\n"
    f"- Python {sys.version}\n"
    f"- TensorFlow {tf.__version__}\n"
    f"- NumPy {np.__version__}\n"
    f"- GPU devices: {len(tf.config.list_physical_devices('GPU'))}"
)

# ---------------------------------------------------------------------------
# Training configuration
# ---------------------------------------------------------------------------
class DynamicTrainingConfig:
    """Configuration class supporting variable input shapes.

    The first call to `detect_input_shape` will populate INPUT_SHAPE based on
    observed dataset maxima.  All other hyperparameters may be tuned from
    outside to reflect recommendations from prior evaluations.
    """

    # Root directory containing Images and Masks (subdirectories or mixed)
    DATA_DIR: Path = Path("/home/rbielski/Atlas_2/Training_Split/Training_Set")

    # The input shape will be set by detect_input_shape; default to None
    INPUT_SHAPE = None

    # Data split configuration
    VALIDATION_SPLIT = 0.15
    SMALL_LESION_THRESHOLD = 100

    # Training schedule
    BATCH_SIZE = 2
    INITIAL_EPOCH = 0
    TOTAL_EPOCHS = 200
    INITIAL_LR = 1e-4
    MIN_LR = 5e-7               # Lower minimum LR for potential late training plateaus
    WARMUP_EPOCHS = 15          # Extended warmup
    MAX_GRAD_NORM = 1.0

    # Model architecture
    BASE_FILTERS = 8
    DROPOUT_RATE = 0.55         # Balanced dropout for robustness
    L2_REG = 1.5e-3             # Moderate L2 regularisation
    MAMBA_DEPTH = 2
    SAM_HEADS = 4

    # Data augmentation
    AUGMENTATION_INTENSITY = 0.5
    SYNTHETIC_LESION_PROB = 0.3
    ROTATION_RANGE = 20         # Slightly larger range than original

    # Loss configuration (based on evaluation recommendations)
    USE_BOUNDARY_LOSS = True
    DICE_LOSS_WEIGHT = 0.4
    BOUNDARY_LOSS_WEIGHT = 0.6
    DEEP_SUPERVISION_WEIGHTS = [0.1, 0.2, 0.3]
    
    #Tweaks
    # Add these defaults anywhere among the other hyperparams:
    DICE_WEIGHT: float = 0.4
    BOUNDARY_WEIGHT: float = 0.6
    # inside class DynamicTrainingConfig:
    RESAMPLE_TO_TARGET = True   # resample both image & mask to INPUT_SHAPE[:-1]


    # Decision threshold for converting logits to binary masks
    DECISION_THRESHOLD = 0.5    # Raise threshold to mitigate over‑segmentation

    # Output directories
    MODEL_DIR = Path("models/dynamic_production")
    CALLBACKS_DIR = Path("callbacks/dynamic_production")

    def __init__(self):
        # Timestamp for saving model checkpoints
        self.timestamp = time.strftime("%Y%m%d_%H%M%S")
        self.MODEL_DIR.mkdir(parents=True, exist_ok=True)
        self.CALLBACKS_DIR.mkdir(parents=True, exist_ok=True)
        # Write config to JSON for reproducibility
        config_dict = {k: v for k, v in self.__dict__.items()
                      if not k.startswith('__') and not callable(v)}
        with open(self.MODEL_DIR / "config.json", "w") as f:
            json.dump(config_dict, f, indent=2, default=str)
        logger.info(
            f"📋 DynamicTrainingConfig initialised:\n"
            f"   Data directory: {self.DATA_DIR}\n"
            f"   Batch size: {self.BATCH_SIZE}\n"
            f"   Warmup epochs: {self.WARMUP_EPOCHS}\n"
            f"   Minimum LR: {self.MIN_LR}\n"
            f"   Dice/boundary weights: {self.DICE_LOSS_WEIGHT}:{self.BOUNDARY_LOSS_WEIGHT}\n"
        )

    @property
    def model_path(self) -> Path:
        return self.MODEL_DIR / f"smart_sota_dynamic_{self.timestamp}.keras"

    @property
    def checkpoint_path(self) -> Path:
        return self.CALLBACKS_DIR / "best_model_dynamic.keras"


# ---------------------------------------------------------------------------
# Custom layers (identical to previous implementations)
# ---------------------------------------------------------------------------
class ResidualConvBlock(layers.Layer):
    """Simplified residual block to avoid CUDNN gradient issues"""
    def __init__(self, filters, kernel_reg=None, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_reg = kernel_reg

    def build(self, input_shape):
        self.conv1 = layers.Conv3D(self.filters, 3, padding='same',
                                   kernel_regularizer=self.kernel_reg)
        self.bn1 = layers.BatchNormalization()
        self.conv2 = layers.Conv3D(self.filters, 3, padding='same',
                                   kernel_regularizer=self.kernel_reg)
        self.bn2 = layers.BatchNormalization()
        self.dropout = layers.SpatialDropout3D(0.1)
        # Always project if channel mismatch
        self.residual_conv = layers.Conv3D(self.filters, 1, padding='same')
        self.residual_bn = layers.BatchNormalization()
        super().build(input_shape)

    def call(self, inputs, training=None):
        x = self.conv1(inputs)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.dropout(x, training=training)
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        # Residual connection
        residual = self.residual_conv(inputs)
        residual = self.residual_bn(residual, training=training)
        return tf.nn.relu(x + residual)

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "kernel_reg": tf.keras.regularizers.serialize(self.kernel_reg)
                           if self.kernel_reg else None
        })
        return config


class VisionMambaBlock(layers.Layer):
    """Efficient vision Mamba block with dynamic input support"""
    def __init__(self, filters, kernel_size=3, expansion=2, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.expansion = expansion

    def build(self, input_shape):
        self.in_conv = layers.Conv3D(
            self.filters * self.expansion,
            1,
            use_bias=False,
            padding='same',
            data_format='channels_last')
        self.spatial_conv = layers.Conv3D(
            self.filters * self.expansion,
            self.kernel_size,
            padding='same',
            use_bias=False,
            data_format='channels_last')
        self.out_conv = layers.Conv3D(
            self.filters, 1,
            padding='same',
            data_format='channels_last')
        self.norm = layers.LayerNormalization()
        self.dropout = layers.SpatialDropout3D(0.1)
        super().build(input_shape)

    def call(self, inputs, training=None):
        x = self.in_conv(inputs)
        x = tf.nn.relu(x)
        x = self.spatial_conv(x)
        x = tf.cast(tf.nn.relu(x), inputs.dtype)
        x = self.dropout(x, training=training)
        x = self.out_conv(x)
        x = self.norm(x)
        return x + inputs

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "expansion": self.expansion
        })
        return config


class SAM2Attention(layers.Layer):
    """Enhanced SAM2 attention with hierarchical memory banks"""
    def __init__(self, filters, heads, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.heads = heads
        self.depth = filters // heads
        if filters % heads != 0:
            raise ValueError("Filters must be divisible by heads")

    def build(self, input_shape):
        self.query = layers.Conv3D(self.filters, 1, data_format='channels_last')
        self.key = layers.Conv3D(self.filters, 1, data_format='channels_last')
        self.value = layers.Conv3D(self.filters, 1, data_format='channels_last')
        self.out_conv = layers.Conv3D(input_shape[-1], 1, data_format='channels_last')
        self.memory_bank = self.add_weight(
            name='memory_bank',
            shape=(1, 1, 1, 1, self.filters),
            initializer='zeros',
            trainable=True
        )
        self.dropout = layers.SpatialDropout3D(0.1)
        super().build(input_shape)

    def call(self, inputs, training=None):
        batch_size = tf.shape(inputs)[0]
        height = tf.shape(inputs)[1]
        width = tf.shape(inputs)[2]
        depth_dim = tf.shape(inputs)[3]
        q = self.query(inputs)
        k = self.key(inputs)
        v = self.value(inputs)
        k = k + self.memory_bank
        v = v + self.memory_bank
        q = self._split_heads_safe(q, batch_size, height, width, depth_dim)
        k = self._split_heads_safe(k, batch_size, height, width, depth_dim)
        v = self._split_heads_safe(v, batch_size, height, width, depth_dim)
        dk = tf.cast(self.depth, q.dtype)
        attn_logits = tf.matmul(q, k, transpose_b=True)
        attn_logits = attn_logits / tf.math.sqrt(dk)
        attn_weights = tf.nn.softmax(attn_logits, axis=-1)
        attn_output = tf.matmul(attn_weights, v)
        attn_output = self._combine_heads_safe(attn_output, batch_size, height, width, depth_dim)
        output = self.out_conv(attn_output)
        output = self.dropout(output, training=training)
        return output + inputs

    def _split_heads_safe(self, x, batch_size, height, width, depth_dim):
        x = tf.reshape(x, [batch_size, height, width, depth_dim, self.heads, self.depth])
        return tf.transpose(x, perm=[0, 4, 1, 2, 3, 5])

    def _combine_heads_safe(self, x, batch_size, height, width, depth_dim):
        x = tf.transpose(x, perm=[0, 2, 3, 4, 1, 5])
        return tf.reshape(x, [batch_size, height, width, depth_dim, self.filters])

    def get_config(self):
        config = super().get_config()
        config.update({
            "filters": self.filters,
            "heads": self.heads
        })
        return config


# ---------------------------------------------------------------------------
# Utility functions for memory monitoring
# ---------------------------------------------------------------------------
def log_memory_usage(stage: str) -> None:
    process = psutil.Process(os.getpid())
    gb_used = process.memory_info().rss / 1024**3
    gpu_mem = []
    try:
        for i in range(4):
            alloc = tf.config.experimental.get_memory_info(f'GPU:{i}')
            gpu_mem.append(f"GPU{i}: {alloc['current']/1e9:.2f}GB")
    except Exception:
        gpu_mem = ["GPU mem tracking failed"]
    try:
        disk_usage = psutil.disk_usage('/')
        disk_free_gb = disk_usage.free / 1024**3
        disk_info = f"Disk: {disk_free_gb:.1f}GB free"
    except Exception:
        disk_info = "Disk: unavailable"
    logger.info(f"Memory at {stage}: CPU={gb_used:.2f}GB | {' | '.join(gpu_mem)} | {disk_info}")


# ---------------------------------------------------------------------------
# Dataset inspection and loading
# ---------------------------------------------------------------------------
from pathlib import Path
import numpy as np
import nibabel as nib
import gc



def detect_input_shape(data_dir: Path) -> tuple:
    """
    Determine the maximum spatial (D,H,W) across NIfTI volumes under `data_dir`,
    then round each dimension UP to the nearest multiple of 16.

    We consider any .nii.gz with at least 3 dims. If none are valid, an error is raised.
    Logs fall back to print() if a global `logger` isn't available.
    """
    import math
    import nibabel as nib

    log = globals().get("logger", None)
    def _info(msg: str):
        if log is not None:
            log.info(msg)
        else:
            print(msg)

    _info("🔍 Detecting input shape from dataset…")

    # Scan all NIfTI files under the root (Images/Masks are fine; we only read headers/shapes)
    image_files = list(data_dir.rglob("*.nii.gz"))
    max_shape = [0, 0, 0]
    invalid = []

    if not image_files:
        raise FileNotFoundError(f"No .nii.gz files found under {data_dir}")

    for f in image_files:
        try:
            img = nib.load(str(f))
            shp = img.shape
            # Need at least 3 spatial dims
            if len(shp) >= 3:
                for i in range(3):
                    max_shape[i] = max(max_shape[i], int(shp[i]))
            else:
                invalid.append(f"{f.name}: shape {shp} has fewer than 3 dims")
        except Exception as e:
            invalid.append(f"{f.name}: failed to load ({e})")

    if all(dim == 0 for dim in max_shape):
        details = ("Issues encountered:\n  - " + "\n  - ".join(invalid)) if invalid else "No details."
        raise RuntimeError(f"No valid 3-D NIfTI files found in {data_dir}. {details}")

    def _ceil16(x: int) -> int:
        return int(math.ceil(x / 16.0) * 16)

    rounded_shape = tuple(_ceil16(dim) for dim in max_shape)

    _info(
        f"📐 Detected max volume dimensions: {tuple(max_shape)} → "
        f"rounded up to: {rounded_shape}"
    )
    return rounded_shape


# --- Replace your existing load_generic_dataset with this version ---
import gc
import numpy as np
import nibabel as nib
from pathlib import Path
import re

def load_generic_dataset(config: DynamicTrainingConfig):
    """
    Loads pairs from:
      <config.DATA_DIR>/Images/*T1w*.nii.gz
      <config.DATA_DIR>/Masks/*mask*.nii.gz

    Pairing:
      image key: strip '_T1w' (before .nii.gz)
      mask  key: strip '_label-..._desc-..._mask' (or trailing '_mask')
    Returns:
      pairs: list[(image_path, mask_path)]
      lesion_presence: np.array of {0,1} per pair (mask has any > 0)
    """
    logger.info("📚 Loading generic dataset (RB pairing rules)...")
    log_memory_usage("dataset_load_start")

    base = config.DATA_DIR
    images_dir = (base / "Images")
    masks_dir  = (base / "Masks")
    if not images_dir.exists() or not masks_dir.exists():
        raise FileNotFoundError(f"Expected subfolders 'Images' and 'Masks' under {base}")

    def img_key(p: Path) -> str:
        s = p.name[:-7] if p.name.endswith(".nii.gz") else p.name
        return s.replace("_T1w", "")

    def msk_key(p: Path) -> str:
        s = p.name[:-7] if p.name.endswith(".nii.gz") else p.name
        s2 = re.sub(r"_label-[^_]+_desc-[^_]+_mask$", "", s)
        if s2 == s:
            s2 = re.sub(r"_mask$", "", s2)
        return s2

    images = sorted([p for p in images_dir.glob("*.nii.gz") if "T1w" in p.name and "mask" not in p.name])
    masks  = sorted([p for p in masks_dir.glob("*.nii.gz")  if "mask" in p.name])

    logger.info(f"✅ Found {len(images)} images under {images_dir}")
    logger.info(f"✅ Found {len(masks)} masks under  {masks_dir}")

    img_map = {img_key(p): p for p in images}
    msk_map = {msk_key(p): p for p in masks}
    keys = sorted(set(img_map).intersection(msk_map.keys()))

    if not keys:
        # Print a few sample names/keys to explain WHY zero pairs
        some_imgs = list(img_map.items())[:5]
        some_msks = list(msk_map.items())[:5]
        logger.error("No image–mask pairs matched. Example keys (image -> file):")
        for k, v in some_imgs:
            logger.error(f"  {k} -> {v.name}")
        logger.error("Example keys (mask -> file):")
        for k, v in some_msks:
            logger.error(f"  {k} -> {v.name}")
        raise RuntimeError("No pairs matched. Check filename patterns / key rules above.")

    pairs = []
    lesion_counts = []
    for k in keys:
        img_p = img_map[k]
        msk_p = msk_map[k]
        try:
            mask_obj = nib.load(str(msk_p))
            has_lesion = bool(np.any(mask_obj.get_fdata() > 0))
            lesion_counts.append(1 if has_lesion else 0)
            pairs.append((img_p, msk_p))
        except Exception as e:
            logger.warning(f"Skipping pair for {k}: {e}")
        finally:
            try:
                del mask_obj
            except:
                pass
            gc.collect()

    logger.info(f"📊 Created {len(pairs)} image–mask pairs")
    if lesion_counts:
        logger.info(f"🧠 Lesion presence: {np.mean(lesion_counts)*100:.2f}%")
    log_memory_usage("dataset_load_end")
    return pairs, np.array(lesion_counts, dtype=np.int32)


def create_stratified_splits(pairs, lesion_presence, batch_size, test_size=0.1):
    """Generate train/validation splits compatible with batch size.

    We adapt the original function to ensure that each split contains a number
    of samples divisible by the batch size.  Stratification is performed
    based on lesion presence.
    """
    total_samples = len(pairs)
    test_samples = math.floor(total_samples * test_size)
    train_samples = total_samples - test_samples
    # Round down to nearest batch size
    test_samples = (test_samples // batch_size) * batch_size
    train_samples = total_samples - test_samples
    # Guarantee at least one batch in each split
    if test_samples < batch_size:
        test_samples = batch_size
        train_samples = total_samples - test_samples
    if train_samples < batch_size:
        train_samples = batch_size
        test_samples = total_samples - train_samples
    logger.info(
        f"🧮 Dataset split: Train={train_samples}"
        f" ({train_samples/total_samples*100:.1f}%), "
        f"Validation={test_samples} ({test_samples/total_samples*100:.1f}%)"
    )
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for train_idx, test_idx in kf.split(pairs, lesion_presence):
        if len(test_idx) >= test_samples:
            test_idx = test_idx[:test_samples]
            break
    train_pairs = [pairs[i] for i in train_idx]
    test_pairs = [pairs[i] for i in test_idx]
    train_lesions = np.mean([lesion_presence[i] for i in train_idx])
    test_lesions = np.mean([lesion_presence[i] for i in test_idx])
    logger.info(
        f"⚖️ Lesion representation: Train={train_lesions*100:.1f}%, "
        f"Validation={test_lesions*100:.1f}%"
    )
    return train_pairs, test_pairs

def pad_and_center_crop(volume: np.ndarray, target_shape: tuple) -> np.ndarray:
    """
    Symmetrically pad (if smaller) or center-crop (if larger) a 3D volume to target_shape.
    Works for both images (float) and masks (binary/float). No interpolation is used.
    """
    assert volume.ndim == 3, f"Expected 3D volume, got {volume.ndim}D"
    z, y, x = volume.shape
    tz, ty, tx = target_shape
    out = volume

    # Center-crop if needed
    if z > tz:
        start = (z - tz) // 2
        out = out[start:start+tz, :, :]
        z = tz
    if y > ty:
        start = (y - ty) // 2
        out = out[:, start:start+ty, :]
        y = ty
    if x > tx:
        start = (x - tx) // 2
        out = out[:, :, start:start+tx]
        x = tx

    # Symmetric pad if needed
    pad_z = max(0, tz - z)
    pad_y = max(0, ty - y)
    pad_x = max(0, tx - x)
    if pad_z or pad_y or pad_x:
        pz0, pz1 = pad_z // 2, pad_z - pad_z // 2
        py0, py1 = pad_y // 2, pad_y - pad_y // 2
        px0, px1 = pad_x // 2, pad_x - pad_x // 2
        out = np.pad(out, ((pz0, pz1), (py0, py1), (px0, px1)), mode="constant", constant_values=0)
    return out

# --- Center-slice helpers (shared crop/pad for image & mask) -----------------
def compute_center_slices(in_shape, out_shape):
    """
    Return input slices that pick the centered sub-volume when cropping, or the
    full axis when padding. Use these slices for BOTH image and mask.
    """
    inD, inH, inW = in_shape
    outD, outH, outW = out_shape
    slices = []
    for i_len, o_len in zip((inD, inH, inW), (outD, outH, outW)):
        if i_len >= o_len:
            start = (i_len - o_len) // 2
            end   = start + o_len
            slices.append(slice(start, end))
        else:
            # padding case: take the whole input on that axis
            slices.append(slice(0, i_len))
    return tuple(slices)  # (sd, sh, sw)

def apply_center_crop_or_pad(vol, in_slices, out_shape):
    """
    Apply the provided input slices, then center-pad into out_shape.
    Use the SAME in_slices for image and mask to guarantee identical transform.
    """
    sub = vol[in_slices[0], in_slices[1], in_slices[2]]
    out = np.zeros(out_shape, dtype=vol.dtype)
    # center place the 'sub' into out
    offs = tuple((o - s) // 2 for s, o in zip(sub.shape, out_shape))
    out[
        offs[0]:offs[0]+sub.shape[0],
        offs[1]:offs[1]+sub.shape[1],
        offs[2]:offs[2]+sub.shape[2],
    ] = sub
    return out.astype(np.float32)


# ---------------------------------------------------------------------------
# Data generator (no augmentations). Only resampling (optional) + center crop/pad.
# ---------------------------------------------------------------------------
import gc
from functools import lru_cache

import nibabel as nib
import numpy as np
import psutil
from scipy.ndimage import zoom
import tensorflow as tf

@lru_cache(maxsize=128)
def _load_vol_canonical(path: str) -> np.ndarray:
    """Load NIfTI as RAS-canonical and return float32 array."""
    img = nib.load(path)
    img = nib.as_closest_canonical(img)  # standardize orientation
    return img.get_fdata().astype(np.float32)

def _load_image(path: str) -> np.ndarray:
    return _load_vol_canonical(path)

def _load_mask_bin(path: str) -> np.ndarray:
    return (_load_vol_canonical(path) > 0.5).astype(np.float32)

def _center_crop_or_pad(vol: np.ndarray, target_shape: tuple[int,int,int]) -> np.ndarray:
    """Center-crop if larger; center-pad with zeros if smaller."""
    assert vol.ndim == 3
    inD, inH, inW = vol.shape
    outD, outH, outW = target_shape
    out = np.zeros(target_shape, dtype=vol.dtype)

    def _slices(in_len, out_len):
        if in_len >= out_len:
            s = (in_len - out_len) // 2
            return slice(s, s + out_len), slice(0, out_len)
        else:
            s = (out_len - in_len) // 2
            return slice(0, in_len), slice(s, s + in_len)

    sD_in, sD_out = _slices(inD, outD)
    sH_in, sH_out = _slices(inH, outH)
    sW_in, sW_out = _slices(inW, outW)
    out[sD_out, sH_out, sW_out] = vol[sD_in, sH_in, sW_in]
    return out.astype(np.float32)

def _normalize_image(img: np.ndarray) -> np.ndarray:
    """Robust [0,1] normalize inside nonzero region."""
    if img.size == 0 or img.max() == 0:
        return np.zeros_like(img, dtype=np.float32)
    brain = img[img > 0]
    if brain.size == 0:
        return np.zeros_like(img, dtype=np.float32)
    p1, p99 = np.percentile(brain, [1, 99])
    img = np.clip(img, p1, p99)
    m, s = brain.mean(), brain.std()
    if s > 0:
        img = (img - m) / s
    mn, mx = img.min(), img.max()
    return ((img - mn) / (mx - mn + 1e-8)).astype(np.float32)

def _prepare_pair(img: np.ndarray, msk: np.ndarray, target_shape: tuple[int,int,int], resample: bool):
    """
    Make image & mask the SAME shape with identical resample/crop/pad steps.
    - resample=True: map current shape -> target_shape (linear for img, nearest for mask)
    - then enforce exact target via centered crop/pad
    """
    assert img.shape == msk.shape, f"pre-prep mismatch: {img.shape} vs {msk.shape}"

    if resample and img.shape != target_shape:
        zoom_factors = tuple(t / s for t, s in zip(target_shape, img.shape))
        img = zoom(img, zoom_factors, order=1, mode="nearest", prefilter=False)
        msk = zoom(msk, zoom_factors, order=0, mode="nearest", prefilter=False)

    if img.shape != target_shape or msk.shape != target_shape:
        img = _center_crop_or_pad(img, target_shape)
        msk = _center_crop_or_pad(msk, target_shape)

    img = _normalize_image(img)
    msk = (msk > 0.5).astype(np.float32)
    return img.astype(np.float32), msk.astype(np.float32)

class DynamicDataGenerator(tf.keras.utils.Sequence):
    """
    1) Load image & mask (canonical orientation)
    2) (Optional) resample both to target_shape
    3) Center-crop/pad both identically
    4) Normalize image (mask stays binary)
    """
    def __init__(self, pairs, config: DynamicTrainingConfig, is_training=True):
        self.pair_paths   = [(str(img), str(mask)) for img, mask in pairs]
        self.batch_size   = config.BATCH_SIZE
        self.target_shape = tuple(config.INPUT_SHAPE[:-1])  # (D,H,W)
        self.config       = config
        self.is_training  = is_training  # kept for API compatibility (not used)
        self.indexes      = np.arange(len(self.pair_paths))

        # Optional resampling to target shape (default: False; set True in config to enable)
        self.resample_to_target = getattr(config, "RESAMPLE_TO_TARGET", False)

        # Light caching if plenty of RAM
        self._cache_enabled = psutil.virtual_memory().available > 50 * 1024**3
        self._volume_cache  = {} if self._cache_enabled else None

        np.random.shuffle(self.indexes)
        logger.info(
            f"🔧 Dynamic data generator: {len(self.pair_paths)} samples, "
            f"target_shape={self.target_shape}, resample_to_target={self.resample_to_target}, "
            f"cache={'enabled' if self._cache_enabled else 'disabled'}"
        )

    def __len__(self):
        return len(self.pair_paths) // self.batch_size

    def on_epoch_end(self):
        np.random.shuffle(self.indexes)
        if self._cache_enabled:
            self._volume_cache.clear()
        gc.collect()

    def __getitem__(self, index):
        batch_idxs = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch      = [self.pair_paths[i] for i in batch_idxs]

        X = np.zeros((len(batch), *self.target_shape, 1), dtype=np.float32)
        y = np.zeros((len(batch), *self.target_shape, 1), dtype=np.float32)

        for i, (img_p, mask_p) in enumerate(batch):
            img, msk = self._load_and_prepare_pair(img_p, mask_p)
            X[i, ..., 0] = img
            y[i, ..., 0] = msk
        return X, y

    def _load_and_prepare_pair(self, img_path: str, mask_path: str):
        img = _load_image(img_path)
        msk = _load_mask_bin(mask_path)

        # Safety: if raw shapes differ, reconcile mask to image shape first
        if img.shape != msk.shape:
            logger.warning(f"Image/Mask shape mismatch before prep: {img.shape} vs {msk.shape} [{img_path}]")
            msk = _center_crop_or_pad(msk, img.shape)

        img, msk = _prepare_pair(img, msk, self.target_shape, self.resample_to_target)

        # Cheap sanity: if mask has positive voxels but image there is all zeros, warn
        if np.sum(msk) > 0 and float(np.sum(img[msk > 0])) == 0.0:
            logger.warning(f"Mask region has zero image signal after prep: {img_path}")

        return img, msk




# ---------------------------------------------------------------------------
# Loss functions and metrics
# ---------------------------------------------------------------------------
# --- Cell A: Losses (drop-in replacement) ---
import tensorflow as tf
from tensorflow.keras.utils import register_keras_serializable

@register_keras_serializable(package="custom")
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    # ensure predictions are in [0,1]
    if y_pred.shape.rank is not None and y_pred.shape[-1] == 1:
        y_pred = tf.nn.sigmoid(y_pred)
    else:
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

    intersection = tf.reduce_sum(y_true * y_pred)
    denom = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return (2.0 * intersection + smooth) / (denom + smooth)

@register_keras_serializable(package="custom")
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def _sobel_3d(t):
    """3D Sobel via separable 1D kernels using conv3d; expects (B,D,H,W,C)."""
    t = tf.cast(t, tf.float32)
    k = tf.constant([1., 2., 1.], dtype=tf.float32)
    d = tf.constant([-1., 0., 1.], dtype=tf.float32)

    def mk(ax):
        if ax == 'x': kx, ky, kz = d, k, k
        elif ax == 'y': kx, ky, kz = k, d, k
        else: kx, ky, kz = k, k, d
        filt = tf.einsum('i,j,k->ijk', kz, ky, kx)  # z,y,x
        filt = filt[:, :, :, tf.newaxis, tf.newaxis] / 32.0  # (Dz,Dy,Dx,1,1)
        return tf.cast(filt, tf.float32)

    fx, fy, fz = mk('x'), mk('y'), mk('z')
    gx = tf.nn.conv3d(t, fx, strides=[1,1,1,1,1], padding='SAME')
    gy = tf.nn.conv3d(t, fy, strides=[1,1,1,1,1], padding='SAME')
    gz = tf.nn.conv3d(t, fz, strides=[1,1,1,1,1], padding='SAME')
    return gx, gy, gz

@register_keras_serializable(package="custom")
def boundary_loss(y_true, y_pred):
    """3D boundary loss compatible with (B,D,H,W,1)."""
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    if y_pred.shape.rank is not None and y_pred.shape[-1] == 1:
        y_pred = tf.nn.sigmoid(y_pred)
    gx_t, gy_t, gz_t = _sobel_3d(y_true)
    gx_p, gy_p, gz_p = _sobel_3d(y_pred)
    gtrue = tf.sqrt(gx_t**2 + gy_t**2 + gz_t**2 + 1e-7)
    gpred = tf.sqrt(gx_p**2 + gy_p**2 + gz_p**2 + 1e-7)
    return tf.reduce_mean(tf.abs(gtrue - gpred))

def make_combined_loss(alpha=0.4, beta=0.6):
    """Factory returns a serializable loss (no lambda capturing external objects)."""
    @register_keras_serializable(package="custom")
    def combined(y_true, y_pred):
        return alpha * dice_loss(y_true, y_pred) + beta * boundary_loss(y_true, y_pred)
    return combined


# ---------------------------------------------------------------------------
# Training pipeline
# ---------------------------------------------------------------------------
# --- Replace your entire Training pipeline block with this version ---
import math
import tensorflow as tf

# make_combined_loss(alpha, beta) and dice_coefficient MUST already be defined
# build_dynamic_model(config), detect_input_shape(config.DATA_DIR),
# DynamicDataGenerator, create_stratified_splits, MemoryMonitoringCallback, etc. must also exist.

def train_dynamic_model(config: DynamicTrainingConfig):
    # Detect input shape (your detect_input_shape already rounds to nearest multiple of 16)
    max_dims = detect_input_shape(config.DATA_DIR)
    config.INPUT_SHAPE = max_dims + (1,)
    logger.info(f"🧭 INPUT_SHAPE set to: {config.INPUT_SHAPE}")

    # Load dataset and create splits
    pairs, lesion_presence = load_generic_dataset(config)
    train_pairs, val_pairs = create_stratified_splits(
        pairs, lesion_presence, batch_size=config.BATCH_SIZE, test_size=config.VALIDATION_SPLIT
    )

    # Generators
    train_gen = DynamicDataGenerator(train_pairs, config, is_training=True)
    val_gen   = DynamicDataGenerator(val_pairs,   config, is_training=False)

    # Model
    with tf.distribute.MirroredStrategy().scope():
        model = build_dynamic_model(config)
        model.summary(print_fn=logger.info)

        # Optimizer & LR
        optimizer = tf.keras.optimizers.Adam(learning_rate=config.INITIAL_LR)

        def lr_schedule(epoch):
            if epoch < config.WARMUP_EPOCHS:
                return config.INITIAL_LR * (epoch + 1) / max(1, config.WARMUP_EPOCHS)
            progress = (epoch - config.WARMUP_EPOCHS) / max(1, config.TOTAL_EPOCHS - config.WARMUP_EPOCHS)
            cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
            return max(config.MIN_LR, config.INITIAL_LR * cosine_decay)

        lr_callback = tf.keras.callbacks.LearningRateScheduler(lr_schedule, verbose=0)
        memory_callback = MemoryMonitoringCallback(log_frequency=1)

        # ✅ Serializable loss object (no lambda) so checkpoints/saves work
        loss_fn = make_combined_loss(alpha=config.DICE_WEIGHT, beta=config.BOUNDARY_WEIGHT)

        # ✅ Save only weights during training; monitor Dice
        ckpt_path = config.checkpoint_path.with_suffix(".weights.h5")
        checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
            filepath=str(ckpt_path),
            monitor="val_dice_coefficient",
            mode="max",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        )

        model.compile(optimizer=optimizer, loss=loss_fn, metrics=[dice_coefficient])

    logger.info("🚀 Starting training...")
    history = model.fit(
        train_gen,
        epochs=config.TOTAL_EPOCHS,
        validation_data=val_gen,
        callbacks=[lr_callback, memory_callback, checkpoint_cb],
        initial_epoch=config.INITIAL_EPOCH,
    )

    # Save final model (full SavedModel /.keras); custom objects should be registered already
    model.save(config.model_path)
    logger.info(f"🏁 Training complete. Model saved to {config.model_path}")
    return history

# If running as a script; in a notebook just call train_dynamic_model(DynamicTrainingConfig())
if __name__ == "__main__":
    cfg = DynamicTrainingConfig()
    _ = train_dynamic_model(cfg)



Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


2025-09-16 14:03:18,932 - tensorflow - INFO - Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
2025-09-16 14:03:18,940 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-09-16 14:03:18,940 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-09-16 14:03:18,941 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlow 2.20.0
- NumPy 2.2.1
- GPU devices: 2
2025-09-16 14:03:18,951 - SmartSOTA_Dynamic - INFO - 📋 DynamicTrainingConfig initialised:
   Data directory: /home/rbielski/Atlas_2/Training_Split/Training_Set
   Batch size: 2
   Warmup epochs: 15
   Minimum LR: 5e-07
   Dice/boundary weights: 0.4:0.6

2025-09-16 14:03:18,952 - SmartSOTA_Dynamic - INFO - 🔍 Detecting input shape from dataset…


Strategy: MirroredStrategy


2025-09-16 14:03:19,403 - SmartSOTA_Dynamic - INFO - 📐 Detected max volume dimensions: (197, 233, 189) → rounded up to: (208, 240, 192)
2025-09-16 14:03:19,404 - SmartSOTA_Dynamic - INFO - 🧭 INPUT_SHAPE set to: (208, 240, 192, 1)
2025-09-16 14:03:19,404 - SmartSOTA_Dynamic - INFO - 📚 Loading generic dataset (RB pairing rules)...
2025-09-16 14:03:19,405 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=13.30GB | GPU mem tracking failed | Disk: 1741.3GB free
2025-09-16 14:03:19,409 - SmartSOTA_Dynamic - INFO - ✅ Found 524 images under /home/rbielski/Atlas_2/Training_Split/Training_Set/Images
2025-09-16 14:03:19,410 - SmartSOTA_Dynamic - INFO - ✅ Found 524 masks under  /home/rbielski/Atlas_2/Training_Split/Training_Set/Masks
2025-09-16 14:07:31,090 - SmartSOTA_Dynamic - INFO - 📊 Created 524 image–mask pairs
2025-09-16 14:07:31,091 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 100.00%
2025-09-16 14:07:31,091 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_end: CPU=13.32G

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


...

190/209 ━━━━━━━━━━━━━━━━━━━━ 32s 2s/step - dice_coefficient: 0.0068 - loss: 1.7489

2025-09-16 14:21:09,206 - SmartSOTA_Dynamic - INFO - Memory at batch_190: CPU=17.90GB | GPU mem tracking failed | Disk: 1741.3GB free


200/209 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - dice_coefficient: 0.0068 - loss: 1.7467

2025-09-16 14:21:26,359 - SmartSOTA_Dynamic - INFO - Memory at batch_200: CPU=17.90GB | GPU mem tracking failed | Disk: 1741.3GB free


209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - dice_coefficient: 0.0069 - loss: 1.7447